In [3]:
import pandas as pd
df = pd.read_csv("data_evolution/dataset_backwash_withtime.csv", index_col="time", parse_dates=True)


In [7]:
df.info()


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 449002 entries, 2024-02-13 11:38:00 to 2024-12-23 06:59:00
Data columns (total 97 columns):
 #   Column                                           Non-Null Count   Dtype  
---  ------                                           --------------   -----  
 0   INTAKE LINE PRESSURE OF HMMF FEED PUMPS          449002 non-null  float64
 1   FIT_001A_FLOW AT INLET HMMF-A                    449002 non-null  float64
 2   FIT_001B_FLOW AT INLET HMMF-B                    449002 non-null  float64
 3   FIT_001C_FLOW AT INLET HMMF-C                    449002 non-null  float64
 4   FIT_001D_FLOW AT INLET HMMF-D                    449002 non-null  float64
 5   FIT_001E_FLOW AT INLET HMMF-E                    449002 non-null  float64
 6   FIT_001F_FLOW AT INLET HMMF-F                    449002 non-null  float64
 7   FIT_001G_FLOW AT INLET HMMF-G                    449002 non-null  float64
 8   FIT_001H_FLOW AT INLET HMMF-H                    449002 non-

In [9]:
import pandas as pd
import numpy as np

# 🔹 Colonnes de production FIT005
#erreur :cols_prod = [c for c in df.columns if "FIT005" in c]
cols_prod = [
    "FIT005A_PERMEATE FLOW",
    "FIT_005B_PERMEATE FLOW",
    "FIT_005C_PERMEATE FLOW",
    "FIT_005D_PERMEATE FLOW",
    "FIT_005E_PERMEATE FLOW",
    "FIT_005F_PERMEATE FLOW",
    "FIT_005G_PERMEATE FLOW",
    "FIT_005H_PERMEATE FLOW"
]

# 🔹 Colonnes de pression différentiel des filtres
cols_dp = [
    "PDIT_001A_DIFF PRESSURE ACROSS HMMF-A",
    "PDIT_001B_DIFF PRESSURE ACROSS HMMF-B",
    "PDIT_001C_DIFF PRESSURE ACROSS HMMF-C",
    "PDIT_001D_DIFF PRESSURE ACROSS HMMF-D",
    "PDIT_001E_DIFF PRESSURE ACROSS HMMF-E",
    "PDIT_001F_DIFF PRESSURE ACROSS HMMF-F",
    "PDIT_001G_DIFF PRESSURE ACROSS HMMF-G",
    "PDIT_001H_DIFF PRESSURE ACROSS HMMF-H",
    "PDIT_001I_DIFF PRESSURE ACROSS HMMF-I",
    "PDIT_001J_DIFF PRESSURE ACROSS HMMF-J"
]

# 🔹 Calculer DP moyen des filtres et ajouter au dataset
df['DP_moyen_filtres'] = df[cols_dp].mean(axis=1)

# 🔹 Initialiser colonnes à ajouter
df['type_backwash'] = 0
df['duree_backwash'] = 0.0
df['intervalle_min'] = 0.0 
df['freq_backwash'] = 0
df['Production_totale'] = df[cols_prod].sum(axis=1)
df['Niveau_reservoir'] = df['LIT_002_FILTER_WATER_STG_TNK']
df = df.drop(columns=cols_prod)

# 🔹 Identifier les périodes de backwash réel
df_bw = df[df['DECISION_BACKWASH_NUM'] > 0].copy()

# 🔹 Grouper par périodes consécutives de même type
df_bw['backwash_group'] = (df_bw['DECISION_BACKWASH_NUM'] != df_bw['DECISION_BACKWASH_NUM'].shift()).cumsum()

# 🔹 Remplir les colonnes calculées dans le dataset original
for grp_id, grp in df_bw.groupby('backwash_group'):
    start_idx = grp.index[0]
    end_idx = grp.index[-1]
    duree = (grp.index[-1] - grp.index[0]).total_seconds() / 60  # en minutes
    type_bw = grp['DECISION_BACKWASH_NUM'].iloc[0]
    
    df.loc[start_idx:end_idx, 'type_backwash'] = type_bw
    df.loc[start_idx:end_idx, 'duree_backwash'] = duree

# 🔹 Calculer intervalle_min entre backwash successifs
backwash_times = df[df['type_backwash'] > 0].index
df.loc[backwash_times, 'intervalle_min'] = backwash_times.to_series().diff().dt.total_seconds() / 60

# 🔹 Calculer freq_backwash par jour
df['date'] = df.index.date
freq_dict = df[df['type_backwash'] > 0].groupby('date').size().to_dict()
df['freq_backwash'] = df['date'].map(freq_dict).fillna(0)
df = df.drop(columns=['date'])

# 🔹 Vérifier
print(df.head())

# 🔹 Sauvegarder
df.to_csv("dataset_backwash_with_calculs.csv")


                     INTAKE LINE PRESSURE OF HMMF FEED PUMPS  \
time                                                           
2024-02-13 11:38:00                                  1.49075   
2024-02-13 11:39:00                                  1.49650   
2024-02-13 11:40:00                                  1.49525   
2024-02-13 11:41:00                                  1.49625   
2024-02-13 11:42:00                                  1.50775   

                     FIT_001A_FLOW AT INLET HMMF-A  \
time                                                 
2024-02-13 11:38:00                     454.837494   
2024-02-13 11:39:00                     454.837494   
2024-02-13 11:40:00                     454.837494   
2024-02-13 11:41:00                     454.837494   
2024-02-13 11:42:00                     454.837494   

                     FIT_001B_FLOW AT INLET HMMF-B  \
time                                                 
2024-02-13 11:38:00                     391.059570   
2024-02-13

In [11]:
df.shape

(449002, 96)

In [13]:
df.head()

,INTAKE LINE PRESSURE OF HMMF FEED PUMPS,FIT_001A_FLOW AT INLET HMMF-A,FIT_001B_FLOW AT INLET HMMF-B,FIT_001C_FLOW AT INLET HMMF-C,FIT_001D_FLOW AT INLET HMMF-D,FIT_001E_FLOW AT INLET HMMF-E,FIT_001F_FLOW AT INLET HMMF-F,FIT_001G_FLOW AT INLET HMMF-G,FIT_001H_FLOW AT INLET HMMF-H,FIT_001I_FLOW AT INLET HMMF-I,...,INDICE_DESEQUILIBRE_FILTRES,DECISION_BACKWASH_NUM,FILTRES_A_BACKWASHER,DP_moyen_filtres,type_backwash,duree_backwash,intervalle_min,freq_backwash,Production_totale,Niveau_reservoir
time,,,,,,,,,,,,,,,,,,,,,
2024-02-13 11:38:00,1.49075,454.837494,391.059570,494.862335,388.456238,371.231262,452.521881,391.543762,452.603119,403.975006,...,0.100190,3,"['HMMF-A', 'HMMF-B', 'HMMF-C', 'HMMF-D', 'HMMF...",0.695475,3,271.0,NaN,734.0,1437.231247,82.893753
2024-02-13 11:39:00,1.49650,454.837494,388.655182,493.031921,388.496887,369.078125,452.521881,391.421875,451.587494,403.812500,...,0.099537,3,"['HMMF-A', 'HMMF-B', 'HMMF-C', 'HMMF-D', 'HMMF...",0.694425,3,271.0,1.0,734.0,1441.015625,82.375000
2024-02-13 11:40:00,1.49525,454.837494,386.984314,493.642059,388.984375,368.468750,452.521881,389.715637,449.596863,401.821869,...,0.099028,3,"['HMMF-A', 'HMMF-B', 'HMMF-C', 'HMMF-D', 'HMMF...",0.693700,3,271.0,1.0,734.0,1440.490616,81.356247
2024-02-13 11:41:00,1.49625,454.837494,386.413788,492.259064,386.181244,367.981262,452.521881,390.040619,450.165619,400.968750,...,0.097855,3,"['HMMF-A', 'HMMF-B', 'HMMF-C', 'HMMF-D', 'HMMF...",0.692450,3,271.0,1.0,734.0,1440.512512,81.487503
2024-02-13 11:42:00,1.50775,454.837494,387.391846,490.632050,387.968750,370.012512,452.521881,388.862488,450.409363,400.359375,...,0.097786,3,"['HMMF-A', 'HMMF-B', 'HMMF-C', 'HMMF-D', 'HMMF...",0.693088,3,271.0,1.0,734.0,1440.774994,81.237503


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 449002 entries, 2024-02-13 11:38:00 to 2024-12-23 06:59:00
Data columns (total 96 columns):
 #   Column                                           Non-Null Count   Dtype  
---  ------                                           --------------   -----  
 0   INTAKE LINE PRESSURE OF HMMF FEED PUMPS          449002 non-null  float64
 1   FIT_001A_FLOW AT INLET HMMF-A                    449002 non-null  float64
 2   FIT_001B_FLOW AT INLET HMMF-B                    449002 non-null  float64
 3   FIT_001C_FLOW AT INLET HMMF-C                    449002 non-null  float64
 4   FIT_001D_FLOW AT INLET HMMF-D                    449002 non-null  float64
 5   FIT_001E_FLOW AT INLET HMMF-E                    449002 non-null  float64
 6   FIT_001F_FLOW AT INLET HMMF-F                    449002 non-null  float64
 7   FIT_001G_FLOW AT INLET HMMF-G                    449002 non-null  float64
 8   FIT_001H_FLOW AT INLET HMMF-H                    449002 non-

In [17]:
import pandas as pd
import numpy as np


# 🔹 Colonnes backwash
cols_backwash = [
    'Niveau_reservoir',
    'DECISION_BACKWASH_NUM',
    'FILTRES_A_BACKWASHER',
    'DP_moyen_filtres',      # si calculée précédemment dans le dataset
    'type_backwash',
    'duree_backwash',
    'intervalle_min',
    'freq_backwash',
    'Production_totale']

# 🔹 Colonnes indicateurs
cols_indices = [
    'INDICE_ENCRASSEMENT_HMMF',
    'DEBIT_ENTREE_HMMF',
    'INDICE_IMPACT_PRODUCTION',
    'INDICE_DESEQUILIBRE_FILTRES'
]

# 🔹 Colonnes ΔP filtres
cols_dp = [c for c in df.columns if "PDIT_001" in c]

# 🔹 Créer dataset final avec les colonnes essentielles
data_prod_finale = df[ cols_backwash + cols_indices + cols_dp].copy()

# 🔹 Vérification
print(data_prod_finale.info())
print(data_prod_finale.head())

# 🔹 Sauvegarder si besoin
data_prod_finale.to_csv("data_prod_finale.csv", index=True)


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 449002 entries, 2024-02-13 11:38:00 to 2024-12-23 06:59:00
Data columns (total 23 columns):
 #   Column                                 Non-Null Count   Dtype  
---  ------                                 --------------   -----  
 0   Niveau_reservoir                       449002 non-null  float64
 1   DECISION_BACKWASH_NUM                  449002 non-null  int64  
 2   FILTRES_A_BACKWASHER                   449002 non-null  object 
 3   DP_moyen_filtres                       449002 non-null  float64
 4   type_backwash                          449002 non-null  int64  
 5   duree_backwash                         449002 non-null  float64
 6   intervalle_min                         449001 non-null  float64
 7   freq_backwash                          449002 non-null  float64
 8   Production_totale                      449002 non-null  float64
 9   INDICE_ENCRASSEMENT_HMMF               449002 non-null  float64
 10  DEBIT_ENTREE_HMMF     